# Optional: speculative decoding with a large target

Before running: **Settings > Accelerator > GPU T4 x2**, and **Settings > Internet > On**.

Tests whether speculative decoding pays off when target steps are expensive. The untrained Qwen2.5-7B-Instruct target runs on the first GPU and the draft on the second.
No fine-tuning, no patient test data, and nothing here depends on the training results.

The 7B model is 14.19 GB, close to a T4's memory, so a run can fail with GPU out of memory. That is reported as a result and saved with any scenarios that finished.
Download `large_target_results.zip` from the Output panel when it finishes.

In [ ]:
!nvidia-smi

In [ ]:
import os
REPO = 'https://github.com/FaatehHaneef/Clinical-records-screener.git'
WORK = '/kaggle/temp/Clinical-records-screener'
if not os.path.exists(WORK):
    !git clone -q {REPO} {WORK}
%cd {WORK}
!git rev-parse HEAD

In [ ]:
!pip install -q -r requirements.txt 2>&1 | tail -3
!python -c "import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.device_count())"

In [ ]:
!python scripts/fetch_resources.py --models-from-lock

In [ ]:
!python -c "from huggingface_hub import snapshot_download; snapshot_download('Qwen/Qwen2.5-7B-Instruct', revision='a09a35458c702b33eeacc393d103063234e8bc28', local_dir='models/qwen2.5-7b', ignore_patterns=['*.bin', '*.gguf', '*.pt', '*.pth', '*.onnx', '.gitattributes']); print('7B ready')"

In [ ]:
!python benchmarks/bench_speculative.py --target models/qwen2.5-7b --draft models/qwen2.5-0.5b --draft-device cuda:1 --label kaggle-2xt4-target7b-draft05b --repeats 3

In [ ]:
!python benchmarks/bench_speculative.py --target models/qwen2.5-7b --draft models/qwen2.5-1.5b --draft-device cuda:1 --label kaggle-2xt4-target7b-draft15b --repeats 3

In [ ]:
import glob, shutil
target = '/kaggle/working/large_target_results'
os.makedirs(target, exist_ok=True)
for path in glob.glob('results/bench_speculative_kaggle-2xt4-target7b-*.json'):
    shutil.copy(path, target)
print(os.listdir(target))
shutil.make_archive('/kaggle/working/large_target_results', 'zip', target)
print('Download large_target_results.zip from the Output panel')